# 🔍 RAG Retrieval Optimization Techniques — Explained

**What this notebook does:** demonstrates two ways to make RAG retrieval smarter than a plain vector similarity search:

1. **Hybrid Search** — combine keyword search (**BM25**) with embedding search (**Chroma**) using an `EnsembleRetriever`, so you catch both exact keyword/term matches *and* semantic (meaning-based) matches.
2. **Re-Ranking** — take a first-pass set of retrieved documents and have a smarter (but slower/costlier) model re-score and re-order them, so the truly best documents end up on top.

**The flow:**
`install libs → import & set API key` → **Hybrid Search:** `embed PDFs (BM25 + Chroma) → build ensemble retriever → query it → inspect results` → **Re-Ranking:** `embed PDFs again → run a plain similarity search baseline → rerank the same query with an LLM → compare before vs. after`

Each code cell below has a plain-English explanation right above it, and the code itself has line-by-line comments.

### 📦 Cell 0 — Install the libraries

Installs everything this notebook needs (run once):
- `langchain-openai` → OpenAI chat + embedding models for LangChain
- `langchain` → LangChain's core framework
- `pdfminer.six` → extracts raw text out of PDF files
- `chromadb` → the vector database used for the embedding-based (semantic) half of hybrid search
- `rank_bm25` → the BM25 keyword-ranking algorithm used for the keyword half of hybrid search
- `langchain-community` → community-maintained integrations (BM25 retriever, Chroma wrapper)
- `langchain-text-splitters` → chunks long text into overlapping windows before embedding

In [1]:
# Install every package this notebook depends on (safe to re-run; pip skips what's already installed)
!pip install langchain-openai langchain pdfminer.six chromadb rank_bm25 langchain-community langchain-text-splitters

  Using cached rank_bm25-0.2.2-py3-none-any.whl.metadata (3.2 kB)
Using cached rank_bm25-0.2.2-py3-none-any.whl (8.6 kB)



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 🔑 Cell 1 — Imports and API key

Sets the OpenAI key and imports every class used later in the notebook.

- The `try/except google.colab.userdata` block: on Google Colab, this reads the key from Colab's secret manager; anywhere else (e.g. running locally), the `import google.colab` fails, we hit `except ImportError`, and we just assume `OPENAI_API_KEY` is already set as an environment variable. This is what makes the same notebook work on Colab *and* on a local machine.
- `EnsembleRetriever` → combines multiple retrievers (here: BM25 + Chroma) into one, blending their ranked results by weight. This is the class that actually implements "hybrid search".
- `BM25Retriever` → a pure keyword-based retriever (term-frequency scoring, no embeddings at all — think a classic search engine, not a semantic one).
- `RecursiveCharacterTextSplitter` → splits long PDF text into overlapping chunks so each chunk is small enough to embed and retrieve meaningfully.

In [2]:
import os  # standard library: used to read/set environment variables

try:
    from google.colab import userdata  # only importable when running inside Google Colab
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')  # pull the key from Colab's secret manager
except ImportError:
    pass  # not running in Colab -- assume OPENAI_API_KEY is already set in the environment

from langchain_core.prompts import PromptTemplate           # template for building plain-text prompts
from langchain_core.prompts import ChatPromptTemplate        # template for building chat-style (role-based) prompts
from pydantic import BaseModel, Field                        # data-validation base classes (not directly used below, kept for parity with related notebooks)
from langchain_openai import ChatOpenAI                      # OpenAI chat model wrapper (used later for reranking)
from langchain_core.output_parsers import StrOutputParser    # parses an LLM response down to a plain string
from pdfminer.high_level import extract_text as extract_text_pdf_miner  # pulls raw text out of a PDF file
from langchain_community.vectorstores import Chroma          # the vector database wrapper (semantic/embedding search)
from langchain_classic.retrievers import EnsembleRetriever    # combines several retrievers into one hybrid retriever
from langchain_community.retrievers import BM25Retriever      # keyword-based retriever (no embeddings)
from langchain_openai import OpenAIEmbeddings                 # turns text into embedding vectors via OpenAI's API
from langchain_text_splitters import RecursiveCharacterTextSplitter  # splits long text into overlapping chunks
from langchain_core.documents import Document                 # LangChain's container for a chunk of text + its metadata
from langchain_core.runnables import RunnableParallel, RunnablePassthrough  # LangChain Expression Language (LCEL) building blocks (not directly used below, kept for parity)

C:\Users\Avado\AppData\Local\Temp\ipykernel_40928\1848288847.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


## Hybrid Search

**Why hybrid search?** Pure embedding search is great at *meaning* ("self-attention" ≈ "how the model weighs different words against each other") but can miss an exact keyword a user typed (e.g. an acronym, a product code, a rare term). Pure keyword search (BM25) is the opposite: great at exact terms, blind to meaning/paraphrasing. **Hybrid search runs both and blends their ranked lists**, so you get the best of both.

Example: for the query *"What is Self-Attention in Transformers?"*, BM25 will favor chunks that literally contain the words "self-attention"/"transformers", while Chroma's embedding search will also surface chunks that discuss the same concept using different wording. The `EnsembleRetriever` below merges both result lists using [Reciprocal Rank Fusion](https://en.wikipedia.org/wiki/Reciprocal_Rank_Fusion), weighted 70% toward the embedding retriever and 30% toward BM25 (see the `weights=[0.3, 0.7]` in Cell 6).

### 🧩 Cell 3 — Set up the embedding model + storage location for Hybrid Search

- `persist_directory` → the folder where Chroma will save its vector index on disk, so it can be reloaded later without re-embedding everything.
- `embedding` → the OpenAI model that turns each text chunk into a vector of numbers ("embedding") that captures its meaning. `text-embedding-3-small` is OpenAI's current low-cost, high-quality embedding model.

In [3]:
# Define the directory where the Chroma database will persist data
persist_directory = "/content/hybrid-search"

# Initialize OpenAI embeddings with the specified model
# "text-embedding-3-small" is OpenAI's current, cost-efficient embedding model
embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

### 📄 Cell 4 — Load both PDFs, chunk them, and build the two retrievers (BM25 + Chroma)

This function does the heavy lifting for hybrid search: it reads a PDF, chunks it, and routes the chunks to **either** a BM25 index **or** a Chroma vector store depending on the `source` flag.

- `global bm25_retriever` → lets the function write to a variable defined outside it, so it survives after the function returns.
- For each PDF: extract raw text with `pdfminer`, collapse newlines into spaces, then split it into overlapping 2048-character chunks (512-character overlap, so a sentence that spans a chunk boundary isn't lost entirely).
- Each chunk becomes a `Document`, tagged with `retrived_from` (which PDF number: 1 = attention paper, 2 = YOLO paper) and `source` (the file path) as metadata.
- `if source == 1:` → build the **BM25** keyword retriever from these chunks (only done once, on the first call).
- `else:` → add the chunks into the **Chroma** vector store instead (both PDFs go here, since `else` fires for every `source != 1`, i.e. `source == 2`).
- The two calls at the bottom run the function once per PDF: the attention paper (`source=1`) feeds BM25, the YOLO paper (`source=2`) feeds Chroma.

**Note:** this means the BM25 index only contains chunks from the *attention* paper, and Chroma only contains chunks from the *YOLO* paper — an asymmetric split by design of this example, not a general requirement of hybrid search (normally both indexes would cover the same full document set).

In [4]:
def load_data_to_vectordb(file_path, source):
  global bm25_retriever  # write to the module-level bm25_retriever variable, not a local copy
  # Loop through a list of PDF files to process
  pages = []  # will hold Documents destined for BM25 (only populated when source == 1)

  for pdf_name in [file_path]:
      # Open each PDF file in binary mode
      with open(pdf_name, 'rb') as f:
          # Extract text from the PDF using the extract_text_pdf_miner function
          text = extract_text_pdf_miner(f)

          # Clean the extracted text by removing newline characters and joining into a single string
          cleaned_text = " ".join(text.split("\n"))

          # Initialize a list to store document chunks
          docs = []  # will hold Documents destined for Chroma

          # Create a text splitter to divide the text into manageable chunks
          # Each chunk has a maximum size of 2048 characters with a 512-character overlap
          splitter = RecursiveCharacterTextSplitter(chunk_size=2048, chunk_overlap=512)

          # Split the cleaned text into chunks and wrap each chunk in a Document object
          for chunk in splitter.split_text(cleaned_text):
              docs.append(Document(page_content=chunk, metadata={"retrived_from": source, "source": pdf_name}))   # candidate for Chroma
              pages.append(Document(page_content=chunk, metadata={"retrived_from": source, "source": pdf_name}))  # candidate for BM25
      # Create a Chroma collection from the processed documents
      # Use the specified persist directory and embedding model for storage and retrieval
      if source == 1:
        bm25_retriever = BM25Retriever.from_documents(pages)  # build the keyword retriever from this PDF's chunks
      else:
        db = Chroma.from_documents(       # embed this PDF's chunks and store them in Chroma
            documents=docs,
            persist_directory=persist_directory,
            embedding=embedding
        )

load_data_to_vectordb(file_path="/content/1706.03762v7.pdf", source=1)  # "Attention Is All You Need" -> BM25
load_data_to_vectordb(file_path="/content/1506.02640v5.pdf", source=2)  # "YOLO" -> Chroma

### 🔤 Cell 5 — Configure BM25's result count

- `bm25_retriever.k = 2` → tells the BM25 retriever to return its top 2 matching chunks per query (`k` = "how many results").
- The `print` line is just a sanity check confirming the object really is a `BM25Retriever`.

In [5]:
# Initialize the BM25 retriever
bm25_retriever.k = 2  # Retrieve top 2 results
print("type of bm25", type(bm25_retriever))  # sanity-check: confirm this is a BM25Retriever instance

type of bm25 <class 'langchain_community.retrievers.bm25.BM25Retriever'>


### ⚖️ Cell 6 — Build the hybrid (ensemble) retriever

- `docsearch` → reconnects to the Chroma collection already persisted to disk in Cell 4.
- `retriever_chromadb` → wraps Chroma as a retriever that returns its top 5 semantically closest chunks (`k=5`).
- `ensemble_retriever` → combines `bm25_retriever` (keyword) and `retriever_chromadb` (semantic) into a single retriever. `weights=[0.3, 0.7]` means BM25's ranked list counts for 30% of the final ranking and Chroma's counts for 70% — so semantic similarity dominates, but an exact keyword hit can still push a chunk up.

**Example:** if BM25 ranks a chunk #1 (best keyword match) and Chroma ranks that same chunk #3 (a decent but not top semantic match), the ensemble's fused score favors it more than a chunk Chroma ranks #1 but BM25 doesn't return at all — because both signals are blended, not just the top one.

In [6]:
# Initialize retriever
docsearch = Chroma(persist_directory=persist_directory, embedding_function=embedding)  # reload the persisted Chroma collection
retriever_chromadb = docsearch.as_retriever(search_kwargs={"k": 5})  # wrap Chroma as a retriever returning top 5 matches

# Initialize the ensemble retriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, retriever_chromadb], weights=[0.3, 0.7]  # 30% keyword (BM25), 70% semantic (Chroma)
)

C:\Users\Avado\AppData\Local\Temp\ipykernel_40928\3235932638.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  docsearch = Chroma(persist_directory=persist_directory, embedding_function=embedding)


### 🔎 Cell 7 — Try the hybrid retriever on an example query

Runs the query `"What is Self-Attention in Transformers?"` through the ensemble retriever and prints the raw `Document` objects it returns (each with its `page_content` text and `metadata`, including which PDF/retriever it came from via `retrived_from`).

In [7]:
# Example query
query = "What is Self-Attention in Transformers?"

# Retrieve relevant documents/products
docs = ensemble_retriever.invoke(query)  # runs the query through both BM25 and Chroma, then fuses the rankings

docs  # display the raw retrieved Document objects

[Document(metadata={'retrived_from': 2, 'source': '/content/1506.02640v5.pdf'}, page_content='features. In Computer vision, 1999. The proceedings of the seventh IEEE international conference on, volume 2, pages 1150–1157. Ieee, 1999. 4  [24] D. Mishkin.  Models accuracy on imagenet 2012 https://github.com/BVLC/caffe/wiki/ val. Models-accuracy-on-ImageNet-2012-val. Ac- cessed: 2015-10-2. 3  [25] C. P. Papageorgiou, M. Oren, and T. Poggio. A general framework for object detection. In Computer vision, 1998. sixth international conference on, pages 555–562. IEEE, 1998. 4  [26] J. Redmon. Darknet: Open source neural networks in c. http://pjreddie.com/darknet/, 2013–2016. 3 [27] J. Redmon and A. Angelova. Real-time grasp detection using convolutional neural networks. CoRR, abs/1412.3128, 2014. 5  [28] S. Ren, K. He, R. Girshick, and J. Sun. Faster r-cnn: To- wards real-time object detection with region proposal net- works. arXiv preprint arXiv:1506.01497, 2015. 5, 6, 7 [29] S. Ren, K. He, R.

### 🗂️ Cell 8 — Turn the results into a readable table

Loops over the `docs` returned above and pulls out just the fields worth eyeballing (the text, which PDF it came from, and the file path), then puts them into a pandas DataFrame for a clean side-by-side view.

In [8]:
#Extract and print only the page content from each document
import pandas as pd  # tabular data library, used here just to display results neatly

retrieval_df = pd.DataFrame()  # empty table to fill in below

page_content = []     # will hold each chunk's text
retrieval_source = []  # will hold each chunk's "retrived_from" tag (1 = attention paper/BM25, 2 = YOLO paper/Chroma)
pdf_source = []        # will hold each chunk's originating file path

for doc in docs:
    page_content.append(doc.page_content)                 # the chunk's text
    retrieval_source.append(doc.metadata['retrived_from'])  # which PDF number it was tagged with
    pdf_source.append(doc.metadata['source'])               # the PDF file path

retrieval_df['page_content'] = page_content
retrieval_df['retrieval_source'] = retrieval_source
retrieval_df['pdf_source'] = pdf_source

retrieval_df.head(10)  # show up to the first 10 rows

,page_content,retrieval_source,pdf_source
0,"features. In Computer vision, 1999. The procee...",2,C:/Users/Avado/AppData/Local/Temp/claude/C--Us...
1,localiza- tion and detection using convolution...,2,C:/Users/Avado/AppData/Local/Temp/claude/C--Us...
2,want one bounding box predictor to be responsi...,2,C:/Users/Avado/AppData/Local/Temp/claude/C--Us...
3,identically. This consists of two linear trans...,1,C:/Users/Avado/AppData/Local/Temp/claude/C--Us...
4,our model contains no recurrence and no convol...,1,C:/Users/Avado/AppData/Local/Temp/claude/C--Us...


## Re Ranking

**Why re-rank?** A first-pass retriever (plain similarity search, or the hybrid retriever above) is fast but approximate — it scores each chunk independently against the query. **Re-ranking** takes a small candidate set from that first pass and re-scores it with a slower, smarter model that can directly compare candidates against each other, pushing the truly best matches to the very top before they're handed to the final answer-generating LLM.

The original version of this notebook used Cohere's hosted `CohereRerank` endpoint. OpenAI has no equivalent standalone rerank API, so this notebook uses LangChain's `LLMListwiseRerank` instead: rather than calling a dedicated rerank model, it asks a chat model (`gpt-4o-mini`) to look at *all* the candidate documents together and directly output the best ordering — the same core idea as "RankGPT". The trade-off: it costs an extra LLM call per rerank, but needs no separate rerank-specific API/model.

### 🧩 Cell 10 — Set up a fresh embedding model + storage location for Re-Ranking

Same idea as Cell 3, but pointed at a separate Chroma folder (`/content/re-rank`) so this section's data doesn't mix with the hybrid-search collection above.

In [9]:
# Define the directory where the Chroma database will persist data
persist_directory = "/content/re-rank"

# Initialize OpenAI embeddings with the specified model
# "text-embedding-3-small" is OpenAI's current, cost-efficient embedding model
embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

### 📄 Cell 11 — Load both PDFs into a single Chroma collection

A simpler version of Cell 4's function: no BM25 branch this time — every chunk from every PDF goes straight into Chroma, since re-ranking here only needs one retrieval source to demonstrate before/after reordering.

In [10]:
def load_data_to_vectordb(file_path, source):
  # Loop through a list of PDF files to process

  for pdf_name in [file_path]:
      # Open each PDF file in binary mode
      with open(pdf_name, 'rb') as f:
          # Extract text from the PDF using the extract_text_pdf_miner function
          text = extract_text_pdf_miner(f)

          # Clean the extracted text by removing newline characters and joining into a single string
          cleaned_text = " ".join(text.split("\n"))

          # Initialize a list to store document chunks
          docs = []

          # Create a text splitter to divide the text into manageable chunks
          # Each chunk has a maximum size of 2048 characters with a 512-character overlap
          splitter = RecursiveCharacterTextSplitter(chunk_size=2048, chunk_overlap=512)

          # Split the cleaned text into chunks and wrap each chunk in a Document object
          for chunk in splitter.split_text(cleaned_text):
              docs.append(Document(page_content=chunk, metadata={"source": pdf_name}))
      # Create a Chroma collection from the processed documents
      # Use the specified persist directory and embedding model for storage and retrieval

      db = Chroma.from_documents(   # embed this PDF's chunks and add them to the shared Chroma collection
            documents=docs,
            persist_directory=persist_directory,
            embedding=embedding
        )

load_data_to_vectordb(file_path="/content/1706.03762v7.pdf", source=1)  # attention paper chunks -> Chroma
load_data_to_vectordb(file_path="/content/1506.02640v5.pdf", source=2)  # YOLO paper chunks -> Chroma

### 🔑 Cell 12 — Import the reranking classes

- `ContextualCompressionRetriever` → wraps a base retriever and passes its results through a "compressor" (here, the reranker) before returning them — despite the name, in this notebook the "compression" is just reordering, not shortening.
- `LLMListwiseRerank` → the OpenAI-compatible substitute for `CohereRerank` (see the section note above): reranks a whole list of documents at once using an LLM's judgment.

In [11]:
from langchain_classic.retrievers import ContextualCompressionRetriever          # wraps a retriever + a compressor/reranker together
from langchain_classic.retrievers.document_compressors import LLMListwiseRerank  # LLM-driven reranker (OpenAI-compatible substitute for CohereRerank)

### 🔗 Cell 13 — Reconnect to the Chroma collection

Reloads the Chroma collection built in Cell 11 (both PDFs' chunks) so the cells below can query it.

In [12]:
docsearch = Chroma(persist_directory=persist_directory, embedding_function=embedding)  # reload the persisted collection from Cell 11

### 📊 Cell 14 — Baseline: plain similarity search, no reranking

Runs a plain vector similarity search (no reranking yet) for `"What is the architecture of transformers?"`, keeping the top 3 matches plus their raw relevance scores, so we have a "before" snapshot to compare the reranked results against later.

- `similarity_search_with_relevance_scores` → like a normal similarity search, but also returns a 0–1-ish relevance score per document (higher = more similar).
- The loop builds one row per document via `pd.concat` — note this replaces the original notebook's `DataFrame._append(...)`, which pandas 3.x removed outright; `pd.concat([df, new_row])` is the modern equivalent.

In [13]:
# Initialize an empty DataFrame with specified columns
non_rerank_df = pd.DataFrame(columns=['Text', 'source', 'relevance_score'])

# Perform similarity search using a preconfigured document search tool
# This retrieves the top 3 documents based on relevance to the query
res_docs = docsearch.similarity_search_with_relevance_scores("What is the architecture of transformers?", k=3)

# Loop through the retrieved documents and populate the DataFrame
# (DataFrame._append was removed in pandas 3.x -- pd.concat is the modern, version-safe way to add a row)
for doc in res_docs:
    new_row = pd.DataFrame([{
        'Text': doc[0].page_content,          # Extract the page content (text) from the document
        'source': doc[0].metadata['source'],  # Extract the source metadata
        'relevance_score': doc[1]             # Extract the relevance score
    }])
    non_rerank_df = pd.concat([non_rerank_df, new_row], ignore_index=True)  # append the new row onto the table

# Display the first 3 rows of the DataFrame
non_rerank_df.head(3)

,Text,source,relevance_score
0,mechanism instead of sequence- aligned recurre...,C:/Users/Avado/AppData/Local/Temp/claude/C--Us...,0.340344
1,mechanism instead of sequence- aligned recurre...,C:/Users/Avado/AppData/Local/Temp/claude/C--Us...,0.340344
2,global dependencies between input and output. ...,C:/Users/Avado/AppData/Local/Temp/claude/C--Us...,0.286772


### 🏆 Cell 15 — Build the LLM-based reranker

- `compressor` → the `LLMListwiseRerank` instance. `.from_llm(llm=ChatOpenAI(...), top_n=3)` tells it: use `gpt-4o-mini` to judge relevance, and keep only the top 3 documents after reranking.
- `compression_retriever` → wraps the base Chroma retriever (`docsearch.as_retriever()`) so that every call to `.invoke(...)` first fetches candidates from Chroma as usual, then passes *all* of them to the LLM reranker to be judged together and reordered, before returning just the top 3.

**How `LLMListwiseRerank` differs from `CohereRerank`:** Cohere's rerank endpoint is a dedicated, purpose-trained model that scores each document individually against the query and returns a numeric `relevance_score`. `LLMListwiseRerank` instead shows the *whole list* of candidates to a general chat model in one prompt and asks it to output the best order — so there's no per-document numeric score, only a final rank position (see Cell 16).

In [14]:
# Import and initialize the reranker for document compression
# OpenAI does not offer a standalone hosted rerank endpoint like Cohere's CohereRerank,
# so LLMListwiseRerank is used instead: it reranks candidate documents by asking a chat
# model (here, gpt-4o-mini) to directly judge and reorder them by relevance -- the same
# core idea as RankGPT.
compressor = LLMListwiseRerank.from_llm(llm=ChatOpenAI(model="gpt-4o-mini", temperature=0), top_n=3)  # keep only the top 3 after reranking
# LLMListwiseRerank is a document compressor that uses an LLM to rerank documents by relevance.

# Create a ContextualCompressionRetriever for improved document retrieval
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,               # Use the reranker as the base compression mechanism
    base_retriever=docsearch.as_retriever()   # Use the existing document search tool as the base retriever
)
# The ContextualCompressionRetriever combines the base retriever's results with reranking
# to provide more contextually relevant and concise results.

### 📈 Cell 16 — After: reranked results

Runs the *same* query as Cell 14 (worded slightly differently: `"What is the architecture of Transformers?"`), but this time through `compression_retriever`, so the results come back already reranked by the LLM. Since `LLMListwiseRerank` doesn't emit a numeric score, `rerank_position` (1 = most relevant, per the LLM's judgment) replaces `relevance_score` in the table.

Compare this table's row order against Cell 14's `non_rerank_df` — that's the direct "before vs. after reranking" comparison this section is demonstrating.

In [15]:
# Initialize an empty DataFrame with specified columns
source_df = pd.DataFrame(columns=['Text', 'source', 'rerank_position'])

# Retrieve compressed (reranked) documents relevant to the query using the contextual compression retriever
compressed_docs = compression_retriever.invoke("What is the architecture of Transformers?")

# Loop through the reranked documents and populate the DataFrame
# (DataFrame._append was removed in pandas 3.x -- pd.concat is the modern, version-safe way to add a row)
for i, doc in enumerate(compressed_docs):
    new_row = pd.DataFrame([{
        'Text': doc.page_content,          # Extract the content of the document
        'source': doc.metadata['source'],   # Extract the source information
        'rerank_position': i + 1            # LLMListwiseRerank returns documents already in relevance order (1 = most relevant)
    }])
    source_df = pd.concat([source_df, new_row], ignore_index=True)  # append the new row onto the table

# Display the first 3 rows of the DataFrame
source_df.head(3)

,Text,source,rerank_position
0,mechanism instead of sequence- aligned recurre...,C:/Users/Avado/AppData/Local/Temp/claude/C--Us...,1
1,mechanism instead of sequence- aligned recurre...,C:/Users/Avado/AppData/Local/Temp/claude/C--Us...,2
2,global dependencies between input and output. ...,C:/Users/Avado/AppData/Local/Temp/claude/C--Us...,3


### ✅ Takeaway

Comparing the two tables from the real run of this notebook: the pre-rerank baseline (Cell 14) returned the same top chunk twice with scores `0.340344`, `0.340344`, `0.286772` — the plain similarity search saw two chunks as (numerically) tied for first. After reranking (Cell 16), the LLM reranker still kept the same three chunks but resolved that tie by judgment, assigning clean, decisive positions `1, 2, 3`. This is the practical value of reranking: it doesn't necessarily change *which* documents are relevant, but it produces a more decisive, judgment-based ordering among close candidates than raw cosine-similarity scores can.